# 01 — Transformações de Lorentz, tempo próprio e paradoxo dos gêmeos

**Capítulo 2 — Relatividade especial**

Este laboratório acompanha o capítulo 2 da apostila. Vamos transformar eventos entre referenciais inerciais, visualizar como os eixos de um diagrama de Minkowski mudam com a velocidade relativa, investigar a relatividade da simultaneidade e comparar tempos próprios.

> **Sua cópia:** este é o notebook canônico do curso. Para experimentar e conservar suas alterações no Colab, use **Arquivo → Salvar uma cópia no Drive**. Você também pode baixar o arquivo `.ipynb`.

## Objetivos

Ao final, você deverá ser capaz de:

1. aplicar as transformações de Lorentz a eventos em $1+1$ dimensões;
2. verificar numericamente e simbolicamente a invariância do intervalo;
3. interpretar a inclinação dos eixos $x'$ e $t'$;
4. explicar por que eventos simultâneos em $S$ geralmente não são simultâneos em $S'$;
5. calcular o tempo próprio de trajetórias por trechos;
6. resolver a versão idealizada do paradoxo dos gêmeos.

Usaremos unidades em que $c=1$. Assim, tempo e distância têm a mesma unidade e velocidades são escritas como $\beta=v/c$, com $|\beta|<1$.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 6),
    "axes.grid": True,
    "grid.alpha": 0.18,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def fator_lorentz(beta):
    """Retorna gamma para beta=v/c, exigindo |beta|<1."""
    beta = np.asarray(beta, dtype=float)
    if np.any(np.abs(beta) >= 1):
        raise ValueError("Uma transformação de Lorentz exige |beta| < 1.")
    return 1 / np.sqrt(1 - beta**2)

def lorentz(x, t, beta):
    """Transforma (x,t) de S para S': x'=gamma(x-beta*t), t'=gamma(t-beta*x)."""
    gamma = fator_lorentz(beta)
    return gamma * (np.asarray(x) - beta * np.asarray(t)), gamma * (np.asarray(t) - beta * np.asarray(x))

def intervalo(dx, dt):
    """Intervalo de Minkowski Delta s² = Delta x² - Delta t²."""
    return np.asarray(dx)**2 - np.asarray(dt)**2

## 1. Transformando um evento

Para $S'$ movendo-se com velocidade $\beta$ no sentido positivo de $x$ em relação a $S$,

$$x'=\gamma(x-\beta t), \qquad t'=\gamma(t-\beta x), \qquad \gamma=\frac{1}{\sqrt{1-\beta^2}}.$$

O acoplamento entre $x$ e $t$ é a origem matemática da relatividade da simultaneidade.

In [ ]:
# Evento E em S e velocidade relativa escolhida
x_E, t_E = 3.0, 5.0
beta_E = 0.60
x_linha, t_linha = lorentz(x_E, t_E, beta_E)

print(f"Em S : E = (x={x_E:.2f}, t={t_E:.2f})")
print(f"Em S': E = (x'={x_linha:.2f}, t'={t_linha:.2f})")
print(f"gamma = {fator_lorentz(beta_E):.3f}")

### O intervalo não muda

Com a convenção da apostila, $\Delta s^2=\Delta x^2-\Delta t^2$. Vamos pedir ao SymPy que faça a álgebra sem substituir números.

In [ ]:
x, t, beta = sp.symbols("x t beta", real=True)
gamma = 1 / sp.sqrt(1 - beta**2)
xp = gamma * (x - beta*t)
tp = gamma * (t - beta*x)
verificacao = sp.simplify(xp**2 - tp**2)
sp.Eq(xp**2 - tp**2, verificacao)

O resultado $x'^2-t'^2=x^2-t^2$ mostra que Lorentz não é uma rotação euclidiana: é uma **rotação hiperbólica** que preserva o intervalo de Minkowski.

## 2. Diagramas de Minkowski e os eixos de $S'$

No gráfico abaixo, $x$ é horizontal e $t$ é vertical. As retas $t=\pm x$ formam o cone de luz.

- o eixo $t'$ é a linha de mundo da origem de $S'$: $x=\beta t$;
- o eixo $x'$ contém eventos com $t'=0$: $t=\beta x$;
- quando $|\beta|$ cresce, os dois eixos se inclinam em direção ao cone de luz;
- os ângulos desenhados são ângulos euclidianos do papel. A geometria física é hiperbólica.

In [ ]:
def diagrama_minkowski(beta=0.6, L=4.0, t0=2.5, ax=None, titulo=None, relatorio=True):
    """Desenha os eixos de S e S', o cone de luz e eventos simultâneos em S."""
    gamma = float(fator_lorentz(beta))
    criado = ax is None
    if criado:
        _, ax = plt.subplots(figsize=(8, 7))

    limite = 6.0
    q = np.linspace(-limite, limite, 400)

    # Cone de luz e eixos do referencial S
    ax.plot(q, q, color="#d59a2e", lw=1.8, label="luz: t = ±x")
    ax.plot(q, -q, color="#d59a2e", lw=1.8)
    ax.axvline(0, color="#687684", lw=1.0, ls=":", label="eixos de S")
    ax.axhline(0, color="#687684", lw=1.0, ls=":")

    # Eixos transformados: x'=0 e t'=0
    ax.plot(beta*q, q, color="#156b8a", lw=2.6, label="eixo t' (x'=0)")
    ax.plot(q, beta*q, color="#a63d40", lw=2.6, label="eixo x' (t'=0)")

    # Dois eventos simultâneos em S
    x_eventos = np.array([-L/2, L/2])
    t_eventos = np.array([t0, t0])
    xp_eventos, tp_eventos = lorentz(x_eventos, t_eventos, beta)
    ax.plot(x_eventos, t_eventos, color="#3c8c5a", lw=2.0, label="simultâneos em S")
    ax.scatter(x_eventos, t_eventos, s=62, color="#245f3b", zorder=5)
    ax.annotate("A", (x_eventos[0], t0), xytext=(-14, 8), textcoords="offset points", weight="bold")
    ax.annotate("B", (x_eventos[1], t0), xytext=(7, 8), textcoords="offset points", weight="bold")

    # Linha de simultaneidade de S' que passa pelo ponto médio (0,t0)
    ax.plot(q, t0 + beta*q, color="#a63d40", ls="--", lw=1.5, alpha=0.8, label="t' constante")

    ax.set(xlim=(-limite, limite), ylim=(-limite, limite), xlabel="posição x", ylabel="tempo t")
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(titulo or rf"Diagrama de Minkowski: $\beta={beta:.2f}$, $\gamma={gamma:.2f}$")
    ax.legend(loc="lower right", fontsize=8, frameon=False)

    if relatorio:
        angulo = np.degrees(np.arctan(beta))
        print(f"beta = {beta:+.2f}   gamma = {gamma:.3f}")
        print(f"inclinação euclidiana do eixo x': {angulo:+.1f}° em relação a x")
        print(f"inclinação euclidiana do eixo t': {angulo:+.1f}° em direção a x")
        print(f"t'_A = {tp_eventos[0]:+.3f}   t'_B = {tp_eventos[1]:+.3f}")
        print(f"Delta t' = t'_B - t'_A = {tp_eventos[1]-tp_eventos[0]:+.3f}")

    if criado:
        plt.show()
    return tp_eventos


### Comparação entre referenciais

Observe simultaneamente quatro velocidades. Em $\beta=0$, os eixos coincidem. À medida que $\beta$ aumenta, os eixos primados aproximam-se das linhas de luz, mas nunca as alcançam para um observador massivo.

In [ ]:
velocidades = [0.0, 0.30, 0.60, 0.85]
fig, axes = plt.subplots(2, 2, figsize=(12, 11), sharex=True, sharey=True)
for b, ax in zip(velocidades, axes.flat):
    diagrama_minkowski(b, ax=ax, titulo=rf"$\beta={b:.2f}$, $\gamma={fator_lorentz(b):.2f}$", relatorio=False)
fig.suptitle("Como os eixos de S' mudam com a velocidade relativa", fontsize=15, y=1.01)
fig.tight_layout()
plt.show()

### Controle de velocidade

No Colab, a linha abaixo aparece como um controle deslizante. Mude $\beta$ e execute novamente a célula. Em Jupyter local, altere diretamente o número. Experimente valores positivos e negativos.

In [ ]:
beta_interativo = 0.60  #@param {type:"slider", min:-0.95, max:0.95, step:0.05}
diagrama_minkowski(beta=beta_interativo)
plt.show()

## 3. Eventos que deixam de ser simultâneos

Os eventos $A$ e $B$ do gráfico satisfazem $\Delta t=0$ e $\Delta x=L$ em $S$. Em $S'$,

$$\Delta t'=\gamma(\Delta t-\beta\Delta x)=-\gamma\beta L.$$

Portanto, para $\beta>0$, o evento de maior $x$ ocorre **antes** em $S'$. Se mudarmos o sinal da velocidade, a ordem se inverte. Como a separação é espacial, nenhum sinal causal pode ligar os eventos; não há violação de causalidade.

In [ ]:
L = 4.0
betas = np.linspace(-0.9, 0.9, 361)
delta_t_linha = -fator_lorentz(betas) * betas * L

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(betas, delta_t_linha, color="#a63d40", lw=2.5)
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.set(xlabel=r"velocidade relativa $\beta$", ylabel=r"$\Delta t'$", title="Eventos simultâneos em S vistos por diferentes referenciais")
plt.show()

## 4. Tempo próprio

Para uma trajetória com velocidade $\beta(t)$,

$$\tau=\int\sqrt{1-\beta(t)^2}\,dt.$$

Se a velocidade é constante durante um intervalo $\Delta t$, então $\Delta\tau=\Delta t/\gamma$. O tempo próprio é o tempo marcado pelo relógio que percorre a linha de mundo.

In [ ]:
def tempo_proprio_segmentos(duracoes, velocidades):
    duracoes = np.asarray(duracoes, dtype=float)
    velocidades = np.asarray(velocidades, dtype=float)
    if duracoes.shape != velocidades.shape:
        raise ValueError("Forneça uma velocidade para cada duração.")
    return np.sum(duracoes / fator_lorentz(velocidades))

# Exemplo da apostila: Delta t=5, Delta x=3
dt, dx = 5.0, 3.0
beta_particula = dx / dt
tau = tempo_proprio_segmentos([dt], [beta_particula])
print(f"beta = {beta_particula:.2f}")
print(f"Delta tau = {tau:.2f}")
print(f"Verificação pelo intervalo: sqrt(dt²-dx²) = {np.sqrt(dt**2-dx**2):.2f}")

## 5. Paradoxo dos gêmeos

Na versão idealizada, a Terra permanece inercial e a nave viaja com velocidade constante $+\beta$ durante metade do tempo terrestre $T$, inverte instantaneamente a velocidade e retorna com $-\beta$.

$$\tau_{\rm Terra}=T, \qquad \tau_{\rm nave}=T\sqrt{1-\beta^2}=\frac{T}{\gamma}.$$

A assimetria está nas linhas de mundo: o viajante muda de referencial no ponto de retorno. A aceleração instantânea é uma idealização; suavizá-la não elimina a diferença de tempo próprio.

In [ ]:
def diagrama_gemeos(beta=0.8, T=10.0):
    fator_lorentz(beta)  # valida beta
    t_ida = np.linspace(0, T/2, 150)
    t_volta = np.linspace(T/2, T, 150)
    x_ida = beta * t_ida
    x_volta = beta * (T - t_volta)
    tau_terra = T
    tau_nave = T / fator_lorentz(beta)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(np.zeros_like(t_ida), t_ida, color="#156b8a", lw=3, label="gêmeo na Terra")
    ax.plot(np.zeros_like(t_volta), t_volta, color="#156b8a", lw=3)
    ax.plot(x_ida, t_ida, color="#a63d40", lw=3, label="gêmeo viajante")
    ax.plot(x_volta, t_volta, color="#a63d40", lw=3)
    ax.scatter([0, beta*T/2, 0], [0, T/2, T], color=["black", "#d59a2e", "black"], zorder=5)
    ax.annotate("retorno", (beta*T/2, T/2), xytext=(8, -4), textcoords="offset points")
    ax.set(xlabel="posição x", ylabel="tempo terrestre t", title=rf"Paradoxo dos gêmeos: $\beta={beta:.2f}$")
    ax.legend(frameon=False)
    plt.show()
    print(f"Idade acumulada na Terra : {tau_terra:.3f}")
    print(f"Idade acumulada na nave  : {tau_nave:.3f}")
    print(f"Diferença de idades       : {tau_terra-tau_nave:.3f}")
    return tau_terra, tau_nave

beta_gemeos = 0.80  #@param {type:"slider", min:0.0, max:0.95, step:0.05}
tempo_total = 10.0  #@param {type:"number"}
diagrama_gemeos(beta_gemeos, tempo_total)

In [ ]:
betas = np.linspace(0, 0.98, 400)
T = 10.0
idade_terra = np.full_like(betas, T)
idade_nave = T / fator_lorentz(betas)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(betas, idade_terra, label="Terra", color="#156b8a", lw=2.5)
ax.plot(betas, idade_nave, label="nave", color="#a63d40", lw=2.5)
ax.fill_between(betas, idade_nave, idade_terra, color="#d59a2e", alpha=0.18, label="diferença de idades")
ax.set(xlabel=r"velocidade da nave $\beta$", ylabel="tempo próprio acumulado", title=rf"Comparação para tempo terrestre total $T={T:g}$")
ax.legend(frameon=False)
plt.show()

## 6. Problemas orientados

Altere os parâmetros nas células e responda primeiro com uma estimativa. Depois use o código para verificar.

### Problema 1 — Relógio de luz

Dois espelhos estão separados por $L$ no referencial do relógio. Um pulso faz uma viagem de ida e volta.

1. Calcule o período próprio $\Delta\tau$.
2. Para um observador que vê o relógio com velocidade $\beta$, use o triângulo formado por metade do percurso e encontre $\Delta t$.
3. Compare com $\Delta t=\gamma\Delta\tau$.

In [ ]:
L_espelhos = 3.0
beta_relogio = 0.80
periodo_proprio = 2 * L_espelhos
periodo_coordenado = fator_lorentz(beta_relogio) * periodo_proprio
meio_periodo_pitagoras = L_espelhos / np.sqrt(1-beta_relogio**2)

print(f"Delta tau = {periodo_proprio:.3f}")
print(f"Delta t por Lorentz = {periodo_coordenado:.3f}")
print(f"Delta t por Pitágoras = {2*meio_periodo_pitagoras:.3f}")

### Problema 2 — Comprimento próprio e simultaneidade

Uma barra está em repouso em $S'$ com comprimento $L_0$. Para medir seu comprimento em $S$, precisamos escolher eventos nas extremidades com o mesmo $t$. Mostre que esses eventos têm tempos $t'$ diferentes e obtenha $L=L_0/\gamma$.

In [ ]:
L0 = 5.0
beta_barra = 0.60
gamma_barra = fator_lorentz(beta_barra)
# Em S, escolhemos t_A=t_B=0. Na transformação inversa isso exige t'_B=-beta*L0.
tp_A, xp_A = 0.0, 0.0
tp_B, xp_B = -beta_barra*L0, L0
x_A = gamma_barra * (xp_A + beta_barra*tp_A)
x_B = gamma_barra * (xp_B + beta_barra*tp_B)

print(f"Em S': Delta t' = {tp_B-tp_A:.3f} (não simultâneos)")
print(f"Em S : L = {x_B-x_A:.3f}")
print(f"L0/gamma = {L0/gamma_barra:.3f}")

### Problema 3 — Inversão da ordem temporal

Considere $A=(0,0)$ e $B=(4,1)$.

1. Classifique a separação.
2. Encontre o valor de $\beta$ para o qual os eventos são simultâneos em $S'$.
3. Escolha uma velocidade maior e verifique que a ordem temporal se inverte.
4. Explique por que isso seria impossível para uma separação temporal.

In [ ]:
dx, dt = 4.0, 1.0
beta_simultaneo = dt/dx
beta_teste = 0.60
_, dt_transformado = lorentz(dx, dt, beta_teste)

print(f"Delta s² = {intervalo(dx,dt):.3f} -> separação espacial")
print(f"beta para Delta t'=0: {beta_simultaneo:.3f}")
print(f"Com beta={beta_teste:.2f}, Delta t'={dt_transformado:.3f}")

### Problema 4 — Tempo próprio máximo

Entre os eventos $P=(0,0)$ e $Q=(0,10)$, compare:

- a trajetória inercial em repouso;
- uma viagem com cinco unidades de tempo a $+0{,}8$ e cinco a $-0{,}8$.

Qual linha de mundo acumula mais tempo próprio?

In [ ]:
tau_inercial = tempo_proprio_segmentos([10.0], [0.0])
tau_viajante = tempo_proprio_segmentos([5.0, 5.0], [0.8, -0.8])
print(f"Linha inercial : tau = {tau_inercial:.3f}")
print(f"Ida e volta    : tau = {tau_viajante:.3f}")
print(f"Diferença      : {tau_inercial-tau_viajante:.3f}")

### Problema 5 — Adição de velocidades e rapidez

A transformação da linha de mundo $x=ut$ fornece

$$u'=\frac{u-\beta}{1-u\beta}.$$

Verifique que $u=1$ continua dando $u'=1$. Depois escreva $u=\tanh\varphi$ e confirme numericamente que rapidezes se somam em transformações sucessivas.

In [ ]:
def transforma_velocidade(u, beta):
    return (u-beta)/(1-u*beta)

print(f"Luz transformada: u' = {transforma_velocidade(1.0, 0.8):.6f}")

beta_1, beta_2 = 0.40, 0.55
rapidez_total = np.arctanh(beta_1) + np.arctanh(beta_2)
beta_por_rapidez = np.tanh(rapidez_total)
beta_por_composicao = (beta_1+beta_2)/(1+beta_1*beta_2)
print(f"beta total por rapidez   = {beta_por_rapidez:.6f}")
print(f"beta total por composição = {beta_por_composicao:.6f}")

## Síntese

- Lorentz mistura coordenadas espaciais e temporais, preservando $\Delta s^2$.
- Os eixos $x'$ e $t'$ inclinam-se em direção ao cone de luz quando $|\beta|$ aumenta.
- Simultaneidade depende do referencial para eventos espacialmente separados.
- O tempo próprio é o comprimento minkowskiano de uma linha de mundo temporal.
- No espaço-tempo plano, entre dois eventos temporalmente separados, a trajetória inercial acumula o maior tempo próprio.

**Para continuar:** escolha seus próprios pares de eventos e velocidades. Tente prever o resultado antes de executar cada célula.